<h1 style='color:#A23B72'>Optimización por Regiones: Diferentes Parámetros para Diferentes Partes</h1>

<p style='color:#b0b0b0'>En este notebook dividimos cada curva en regiones (segmentos) y aplicamos diferentes configuraciones gaussianas a cada región. Esto refleja la idea de que diferentes partes de la misma curva pueden tener características distintas que requieren parámetros diferentes.</p>

## 1. Motivación: Análisis Regional

### El Problema

Una curva puede tener **heterogeneidad interna**:
- Inicio suave → pocas campanas
- Centro con picos → más campanas
- Final con variación → método adaptativo

**Idea**: Dividir la curva en K regiones y optimizar cada región independientemente.

### Analogía: Análisis de Señales Cardíacas

En PTB-XL (notebook del proyecto), el ciclo cardíaco se divide en:
- **P wave**: Onda suave → polinomio
- **QRS complex**: Pico agudo → gaussiana
- **T wave**: Forma asimétrica → spline

Aquí aplicamos la misma filosofía: **regiones distintas, parámetros distintos, mejor desempeño global.**

### Objetivos

1. Desarrollar método automático de segmentación
2. Optimizar cada región independientemente
3. Evaluar si desempeño regional > desempeño global
4. Identificar cuándo tiene sentido usar análisis regional

## 2. Setup y Importar Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25
})

PALETTE = {
    'gaussiana': '#A23B72',
    'datos': '#5BC0EB',
    'referencia': '#e0e0e0',
    'region1': '#FF6B6B',
    'region2': '#4ECDC4',
    'region3': '#FFE66D'
}

ROOT = Path.cwd().parent
DATA_T = ROOT / 'datos' / 'target'
DATA_P = ROOT / 'datos' / 'pixel_curves'
RESULTADOS_DIR = ROOT / 'metodo_gaussiano' / 'resultados'

print('✓ Setup completado')

## 3. Métodos de Segmentación

In [ ]:
def leer_target(cid):
    """Lee curva target."""
    return pd.read_csv(
        DATA_T / f'curve_{cid:04d}.txt',
        header=None,
        names=['x', 'y']
    ).sort_values('x').reset_index(drop=True)

def segmentar_uniforme(curva, n_regiones=3):
    """Divide curva en n regiones de igual ancho en el eje X.
    
    Retorna lista de (x_seg, y_seg) para cada región.
    """
    x = curva['x'].values
    y = curva['y'].values
    
    x_min, x_max = x.min(), x.max()
    ancho_region = (x_max - x_min) / n_regiones
    
    regiones = []
    for i in range(n_regiones):
        x_inicio = x_min + i * ancho_region
        x_fin = x_min + (i + 1) * ancho_region
        
        mask = (x >= x_inicio) & (x <= x_fin)
        if mask.sum() > 0:
            x_seg = x[mask]
            y_seg = y[mask]
            regiones.append((x_seg, y_seg, f'Región {i+1} [{x_inicio:.2f}, {x_fin:.2f}]'))
    
    return regiones

def segmentar_por_picos(curva, n_regiones=3):
    """Divide curva detectando picos y segmentando alrededor de ellos.
    
    Estrategia:
    - Detecta n_regiones-1 picos principales
    - Divide en regiones usando picos como frontera
    """
    x = curva['x'].values
    y = curva['y'].values
    
    y_centered = y - y.mean()
    picos, _ = find_peaks(y_centered, prominence=0.1*(y.max() - y.min()))
    
    # Seleccionar n-1 picos principales (por altura)
    if len(picos) >= n_regiones - 1:
        idx_top = np.argsort(y_centered[picos])[-(n_regiones-1):]
        fronteras_idx = sorted(picos[idx_top])
    else:
        # Si no hay suficientes picos, usar segmentación uniforme
        return segmentar_uniforme(curva, n_regiones)
    
    # Crear regiones
    regiones = []
    indices_regiones = [0] + fronteras_idx.tolist() + [len(x)-1]
    
    for i in range(len(indices_regiones) - 1):
        idx_start = indices_regiones[i]
        idx_end = indices_regiones[i + 1]
        
        x_seg = x[idx_start:idx_end+1]
        y_seg = y[idx_start:idx_end+1]
        
        if len(x_seg) > 2:
            regiones.append((x_seg, y_seg, f'Región {i+1}'))
    
    return regiones

print('✓ Funciones de segmentación definidas')

## 4. Visualización de Segmentaciones

In [ ]:
# Ejemplo: Visualizar segmentaciones
curva_ejemplo = leer_target(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Segmentación uniforme
ax = axes[0]
regiones_uniforme = segmentar_uniforme(curva_ejemplo, n_regiones=3)
colors_uniforme = ['#FF6B6B', '#4ECDC4', '#FFE66D']

for (x_seg, y_seg, label), color in zip(regiones_uniforme, colors_uniforme):
    ax.plot(x_seg, y_seg, linewidth=2.5, label=label, color=color)

ax.set_title('Segmentación Uniforme (3 regiones)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Segmentación por picos
ax = axes[1]
regiones_picos = segmentar_por_picos(curva_ejemplo, n_regiones=3)

for (x_seg, y_seg, label), color in zip(regiones_picos, colors_uniforme):
    ax.plot(x_seg, y_seg, linewidth=2.5, label=label, color=color)

ax.set_title('Segmentación por Picos (3 regiones)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de Métodos de Segmentación', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\n✓ Segmentación uniforme: {len(regiones_uniforme)} regiones')
print(f'✓ Segmentación por picos: {len(regiones_picos)} regiones')

## 5. Optimización Regional

In [ ]:
def suma_gaussianas(x, *params):
    """Suma de n campanas gaussianas."""
    n_campanas = (len(params) - 1) // 3
    c = params[-1]
    y = np.full_like(x, c, dtype=float)
    
    for i in range(n_campanas):
        A = params[3*i]
        mu = params[3*i + 1]
        sigma = params[3*i + 2]
        y += A * np.exp(-(x - mu)**2 / (2 * sigma**2 + 1e-9))
    
    return y

def optimizar_region(x_region, y_region, n_campanas=2):
    """Optimiza una región individual.
    
    Retorna: (y_pred, params, r2, mensaje)
    """
    
    # Inicialización simple
    y_centered = y_region - y_region.mean()
    picos, _ = find_peaks(y_centered)
    
    n_campanas = min(n_campanas, len(picos) if len(picos) > 0 else 1)
    n_campanas = max(1, n_campanas)
    
    p0 = []
    for _ in range(n_campanas):
        p0 += [y_region.max() - y_region.min(), x_region.mean(), (x_region.max() - x_region.min())/4]
    p0.append(y_region.mean())
    
    try:
        popt, _ = curve_fit(suma_gaussianas, x_region, y_region, p0=p0, maxfev=2000)
        y_pred = suma_gaussianas(x_region, *popt)
        
        ss_res = np.sum((y_region - y_pred)**2)
        ss_tot = np.sum((y_region - y_region.mean())**2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        
        return y_pred, popt, r2, 'OK'
    
    except Exception as e:
        return y_region, p0, 0.0, f'Error: {str(e)[:30]}'

print('✓ Función de optimización regional definida')

## 6. Comparación: Global vs Regional

In [ ]:
# Cargar configuración óptima global del piloto
df_ganadores = pd.read_csv(RESULTADOS_DIR / 'piloto_configuraciones_ganadoras.csv')

# Ejemplo: Curva 1
curva_id = 1
curva = leer_target(curva_id)
x_curva = curva['x'].values
y_curva = curva['y'].values

# Obtener configuración global óptima
config_global = df_ganadores[df_ganadores['curva_id'] == curva_id].iloc[0]
n_campanas_global = config_global['n_campanas']
r2_global = config_global['r2']

print(f'Curva {curva_id}: Análisis Global vs Regional')
print(f'='*70)
print(f'\nConfigración GLOBAL (óptima):') 
print(f'  n_campanas: {n_campanas_global}')
print(f'  Método: {config_global["metodo_init"]}')
print(f'  R²: {r2_global:.4f}')

# Segmentar y optimizar regionalmente
regiones = segmentar_uniforme(curva, n_regiones=3)

print(f'\nConfigración REGIONAL (3 regiones):') 
resultados_regional = []
r2_total_regional = 0

for i, (x_seg, y_seg, label) in enumerate(regiones):
    n_camp_region = max(1, int(n_campanas_global / 3 * (i+1)))
    y_pred, params, r2_region, msg = optimizar_region(x_seg, y_seg, n_campanas=n_camp_region)
    
    resultados_regional.append({
        'region': i+1,
        'n_campanas': n_camp_region,
        'r2': r2_region,
        'y_pred': y_pred,
        'x_seg': x_seg,
        'y_seg': y_seg
    })
    
    print(f'  Región {i+1}: n_campanas={n_camp_region}, R²={r2_region:.4f}')
    r2_total_regional += r2_region

r2_promedio_regional = r2_total_regional / len(regiones)
print(f'\n  R² Promedio Regional: {r2_promedio_regional:.4f}')
print(f'\nComparación:')
print(f'  R² Global:    {r2_global:.4f}')
print(f'  R² Regional:  {r2_promedio_regional:.4f}')
print(f'  Diferencia:   {r2_promedio_regional - r2_global:+.4f}')

## 7. Visualización: Global vs Regional

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Panel izquierdo: Global
ax = axes[0]
ax.plot(x_curva, y_curva, color=PALETTE['referencia'], lw=2, label='Target', zorder=2)
ax.plot(x_curva, y_curva, color=PALETTE['gaussiana'], lw=1.5, alpha=0.7, 
        label=f'Global (n={n_campanas_global}, R²={r2_global:.4f})', linestyle='--', zorder=3)
ax.set_title('Enfoque GLOBAL', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel derecho: Regional
ax = axes[1]
ax.plot(x_curva, y_curva, color=PALETTE['referencia'], lw=2, label='Target', zorder=2)

colors_region = ['#FF6B6B', '#4ECDC4', '#FFE66D']
for res, color in zip(resultados_regional, colors_region):
    ax.plot(res['x_seg'], res['y_pred'], color=color, lw=2,
            label=f"Región {res['region']} (R²={res['r2']:.4f})", zorder=3)

ax.set_title('Enfoque REGIONAL (3 regiones)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Curva {curva_id}: Global vs Regional | Mejora: {r2_promedio_regional - r2_global:+.4f}', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('✓ Visualización completada')

## 8. Conclusiones y Casos de Uso

### Cuándo Usar Análisis Regional

| Característica | Global | Regional |
|---|---|---|
| **Complejidad** | Simple | Más complejo |
| **Tiempo de cómputo** | Rápido | Más lento (K × más) |
| **R² esperado** | Bueno (0.95+) | Mejor si heterogéneo |
| **Interpretabilidad** | Alta | Media (K regiones) |
| **Mejor para** | Curvas homogéneas | Curvas con estructura local |

### Heurística de Decisión

- **Si σ(n_picos por región) es ALTA**: Usar regional
- **Si R² global ya es > 0.99**: No vale la pena regional
- **Si K > 5 regiones necesarias**: Reconsiderar, quizás método híbrido

### Integración con PTB-XL

Este enfoque regional es exactamente lo que PTB-XL hace:
- P wave: Polinomio (suave)
- QRS: Gaussiana (pico agudo)
- T wave: Spline (asimétrico)

**Próximos pasos**: Aplicar segmentación automática y optimización regional a todo el dataset de 500 curvas.